# Module 5 : du sac de mots au fine-tuning, comparés *(optionnel)*

On reprend la tâche du **J3** (deviner si une critique Allociné est *positive* ou *négative*) et on met en concurrence **7 approches**, du plus simple au plus sophistiqué, pour situer le **fine-tuning** parmi les autres. Toutes sont évaluées de la même façon : **accuracy** et **F1** (vus au J2) sur le même jeu de test.

| # | Approche | Idée |
|---|---|---|
| 1 | Comptes de mots + régression logistique | sac de mots brut |
| 2 | TF-IDF (vocabulaire nettoyé) + régression logistique | mots pondérés, vocabulaire réduit |
| 3 | TF-IDF (vocabulaire nettoyé) + ridge | idem, autre classifieur |
| 4 | DistilCamemBERT **gelé** + ridge | embeddings d'un transformer, sans adaptation |
| 5 | DistilCamemBERT **fine-tuning partiel** + ridge | on dégèle les couches hautes |
| 6 | DistilCamemBERT **fine-tuning complet** + ridge | on adapte tout l'encodeur |
| 7 | **LLM** *chain-of-thought* (Claude Haiku) | on demande au modèle de raisonner puis trancher |

> Notebook **optionnel** et avancé. Active le **GPU** (Colab : `Exécution` > `Modifier le type d'exécution` > **GPU T4**). Le modèle 7 passe par une API (clé non fournie aux étudiants, voir plus bas).

In [ ]:
# Sur Colab : installation des paquets (datasets, spacy et anthropic ne sont pas toujours préinstallés)
!pip install -q transformers datasets spacy anthropic
!python -m spacy download fr_core_news_sm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Matériel détecté :", "GPU (cuda)" if device == "cuda" else "CPU seulement (lent : pense à activer le GPU)")

## Les données et la règle du jeu

On charge le corpus Allociné (même source qu'au J3), on convertit l'étiquette en entier (`0` = négatif, `1` = positif) et on sous-échantillonne pour que tout tienne en quelques minutes. Un seul **jeu de test** sert à comparer tous les modèles.

On garde les scores de chaque modèle dans un dictionnaire `results` : on les affichera tous ensemble dans un tableau et un graphique à la fin.

In [ ]:
URL = "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/data/critiques_films"
train = pd.read_csv(f"{URL}/critiques_films_train.csv")
test = pd.read_csv(f"{URL}/critiques_films_test.csv")

label_map = {"négatif": 0, "positif": 1}
train["label"] = train["polarite"].map(label_map)
test["label"] = test["polarite"].map(label_map)

# Sous-échantillonnage (augmente si tu as le temps / un bon GPU)
train = train.sample(2000, random_state=42).reset_index(drop=True)
test = test.sample(800, random_state=42).reset_index(drop=True)
y_train, y_test = train["label"].to_numpy(), test["label"].to_numpy()
print(f"Train : {len(train)} | Test : {len(test)}")

# On collecte ici les scores de chaque modèle pour le tableau / graphique final.
results = {}
def evaluate(name, y_true, y_pred, n=None):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    results[name] = {"accuracy": round(acc, 3), "f1": round(f1, 3), "n_test": n or len(y_true)}
    print(f"{name:46} acc={acc:.3f}  f1={f1:.3f}")

## Modèle 1 : comptes de mots + régression logistique

Le point de départ historique du text mining : on compte les mots (sac de mots, sans nettoyage ni pondération) et on entraîne une régression logistique.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

count_vectorizer = CountVectorizer()                 # tokenisation basique, comptes bruts
X_train = count_vectorizer.fit_transform(train["text"])
X_test = count_vectorizer.transform(test["text"])

model_1 = LogisticRegression(max_iter=1000).fit(X_train, y_train)
evaluate("1. Comptes + régression log", y_test, model_1.predict(X_test))

## Modèles 2 & 3 : nettoyer le vocabulaire (comme au J3) puis TF-IDF

Avant de pondérer les mots en TF-IDF, on **réduit le vocabulaire**, exactement dans l'esprit du notebook `01-text-mining-concepts`.

**À quoi sert spaCy ici ?** `spaCy` est une bibliothèque de traitement du langage. On charge `fr_core_news_sm`, un petit modèle **pré-entraîné pour le français** (`fr` = français, `news` = entraîné sur de l'actualité, `sm` = *small*, léger et rapide). Il sait découper une phrase en mots et, pour chaque mot, donner :
- son **lemme** (forme de base) : « adoré », « adore », « adorais » deviennent tous `adorer` ;
- s'il s'agit d'un **stopword** (mot outil : « le », « de », « et »...) ;
- s'il est uniquement **alphabétique** (pour jeter ponctuation et chiffres).

On l'utilise pour transformer chaque critique en une liste de mots **normalisés et utiles**. `disable=["parser", "ner"]` coupe deux composants dont on n'a pas besoin (analyse grammaticale et détection de noms propres), pour aller plus vite. Ensuite, `TfidfVectorizer` pondère ces mots et applique un **filtrage de fréquence** (`min_df` / `max_df`) qui élimine les mots trop rares ou trop courants.

In [ ]:
import spacy
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])

def clean_texts(texts):
    """Lemmatise et retire stopwords / ponctuation / mots <= 2 lettres."""
    cleaned = []
    for doc in nlp.pipe(texts, batch_size=64):
        lemmas = [token.lemma_.lower() for token in doc
                  if token.is_alpha and not token.is_stop and len(token.lemma_) > 2]
        cleaned.append(" ".join(lemmas))
    return cleaned

train_clean = clean_texts(train["text"].tolist())
test_clean = clean_texts(test["text"].tolist())
print("Exemple nettoyé :", train_clean[0][:120], "...")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# min_df=5 : mot présent dans >= 5 critiques ; max_df=0.5 : on jette les mots trop fréquents
tfidf = TfidfVectorizer(min_df=5, max_df=0.5)
X_train = tfidf.fit_transform(train_clean)
X_test = tfidf.transform(test_clean)
print("Vocabulaire après réduction :", len(tfidf.get_feature_names_out()), "mots")

**Modèle 2** : TF-IDF + régression logistique.

In [ ]:
model_2 = LogisticRegression(max_iter=1000).fit(X_train, y_train)
evaluate("2. TF-IDF nettoyé + régression log", y_test, model_2.predict(X_test))

**Modèle 3** : mêmes features, mais une **régression ridge** (une régression linéaire régularisée, sobre et rapide, qui se comporte bien quand il y a beaucoup de features comme ici).

In [ ]:
from sklearn.linear_model import RidgeClassifier

model_3 = RidgeClassifier().fit(X_train, y_train)
evaluate("3. TF-IDF nettoyé + ridge", y_test, model_3.predict(X_test))

## Modèles 4 à 6 : DistilCamemBERT comme extracteur de features

Idée commune : un transformer transforme chaque critique en un **vecteur** (son *embedding*, ici la moyenne des états cachés). On entraîne ensuite une **régression ridge** sur ces vecteurs.

Ce qui change d'un modèle à l'autre, c'est **combien on adapte l'encodeur** à notre tâche avant d'extraire les vecteurs :
- **4. gelé** : on ne touche pas au transformer (transfert pur) ;
- **5. fine-tuning partiel** : on ré-entraîne seulement ses **couches hautes** ;
- **6. fine-tuning complet** : on ré-entraîne **tout** l'encodeur.

Plus on adapte, meilleurs (en principe) sont les vecteurs pour *cette* tâche, mais plus c'est coûteux.

In [ ]:
from transformers import AutoTokenizer, AutoModel
MODEL_NAME = "cmarkea/distilcamembert-base"   # DistilCamemBERT : un CamemBERT léger et rapide
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

@torch.no_grad()
def embed(encoder, texts, batch_size=32, max_length=256):
    """Transforme chaque texte en un vecteur = moyenne (pondérée par le masque) des états cachés."""
    encoder = encoder.to(device).eval()
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                          max_length=max_length, return_tensors="pt").to(device)
        hidden = encoder(**batch).last_hidden_state            # (n, tokens, dim)
        mask = batch["attention_mask"].unsqueeze(-1).float()
        mean = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
        vectors.append(mean.cpu().numpy())
    return np.vstack(vectors)

### Modèle 4 : encodeur gelé + ridge

In [ ]:
frozen_encoder = AutoModel.from_pretrained(MODEL_NAME)
X_train = embed(frozen_encoder, train["text"].tolist())
X_test = embed(frozen_encoder, test["text"].tolist())

model_4 = RidgeClassifier().fit(X_train, y_train)
evaluate("4. DistilCamemBERT gelé + ridge", y_test, model_4.predict(X_test))

### Fine-tuning (modèles 5 et 6)

On définit un utilitaire `fine_tune` qui prend **explicitement** les textes et étiquettes d'entraînement (pas de variable cachée), fine-tune un DistilCamemBERT de classification, puis renvoie son **encodeur adapté** (pour en réextraire des embeddings). Le paramètre `n_layers` contrôle l'ampleur :
- `n_layers=1` : seules la **dernière couche** et la tête sont ré-entraînées (partiel) ;
- `n_layers=None` : **tout** est ré-entraîné (complet).

*Pourquoi c'est rapide (~1 min) ?* Petit jeu (2000 exemples), **1 époque**, modèle **distillé** et léger, sur GPU : à peine ~125 pas de gradient. C'est volontairement modeste pour une démo.

In [ ]:
import re
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from datasets import Dataset

def freeze_except_top(model, n_layers):
    """Ne laisse entraînables que les `n_layers` couches hautes de l'encodeur + la tête."""
    if n_layers is None:
        return  # fine-tuning complet : tout reste entraînable
    names = [name for name, _ in model.named_parameters()]
    indices = sorted({int(match.group(1)) for name in names
                      for match in [re.search(r"\.layer\.(\d+)\.", name)] if match})
    trainable = set(indices[-n_layers:])
    for name, param in model.named_parameters():
        match = re.search(r"\.layer\.(\d+)\.", name)
        is_top_layer = bool(match) and int(match.group(1)) in trainable
        is_head = any(key in name for key in ["classifier", "pooler"])
        param.requires_grad = is_top_layer or is_head

def fine_tune(texts, labels, n_layers):
    """Fine-tune un DistilCamemBERT et renvoie son encodeur adapté.

    texts, labels : données d'entraînement passées EXPLICITEMENT (aucune variable globale).
    n_layers      : nb de couches hautes à dégeler (None = tout l'encodeur).
    """
    data = Dataset.from_dict({"text": list(texts), "label": list(labels)})
    data = data.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=256),
                    batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    freeze_except_top(model, n_layers)
    args = TrainingArguments(output_dir="ft", num_train_epochs=1,
                             per_device_train_batch_size=16, learning_rate=2e-5,
                             weight_decay=0.01, logging_steps=50, report_to="none")
    Trainer(model=model, args=args, train_dataset=data,
            data_collator=DataCollatorWithPadding(tokenizer)).train()
    return model.base_model  # l'encodeur adapté

### Modèle 5 : fine-tuning partiel + ridge

In [ ]:
partial_encoder = fine_tune(train["text"].tolist(), y_train, n_layers=1)
X_train = embed(partial_encoder, train["text"].tolist())
X_test = embed(partial_encoder, test["text"].tolist())

model_5 = RidgeClassifier().fit(X_train, y_train)
evaluate("5. DistilCamemBERT partiel + ridge", y_test, model_5.predict(X_test))

### Modèle 6 : fine-tuning complet + ridge

In [ ]:
full_encoder = fine_tune(train["text"].tolist(), y_train, n_layers=None)
X_train = embed(full_encoder, train["text"].tolist())
X_test = embed(full_encoder, test["text"].tolist())

model_6 = RidgeClassifier().fit(X_train, y_train)
evaluate("6. DistilCamemBERT complet + ridge", y_test, model_6.predict(X_test))

## Modèle 7 : un LLM qui raisonne (*chain-of-thought*)

Dernière approche, radicalement différente : on **ne l'entraîne pas du tout**. On demande à un grand modèle de langue (ici **Claude Haiku**) de **raisonner étape par étape** sur la critique (le ton, les mots positifs / négatifs, l'avis global) avant de conclure. C'est le *chain-of-thought* : forcer le modèle à « réfléchir à voix haute » améliore souvent ses décisions.

**Pourquoi une clé n'est pas distribuée.** Chaque critique = un appel API facturé. Sur tout le jeu de test, cela fait des **centaines d'appels** : pour éviter qu'une clé partagée parte en fumée (boucles, relances, oublis), **on ne donne pas de clé aux étudiants**. C'est l'**instructeur** qui lance la démo avec **sa propre clé**.

On définit d'abord les fonctions. **L'appel API n'est pas exécuté ici** (voir la cellule suivante).

In [ ]:
import anthropic

# La clé n'est PAS distribuée aux étudiants (voir explication ci-dessus).
# Seul l'instructeur la renseigne, le temps de la démo.
api_key = "CLE_INSTRUCTEUR"
client = anthropic.Anthropic(api_key=api_key)

SYSTEM_PROMPT = "Annotateur de sentiment de critiques de films en français."

def cot_prompt(review):
    # Prompt court (entrée) + raisonnement bref (sortie) pour limiter le coût.
    return (f'Critique : "{review[:800]}"\n'
            "En une phrase, repère les indices positifs/négatifs, puis conclus.\n"
            "Dernière ligne au format exact : RÉPONSE: positif  ou  RÉPONSE: négatif")

def parse_label(text):
    lowered = text.lower()
    pos, neg = lowered.rfind("positif"), lowered.rfind("négatif")
    if pos == neg:        # aucun des deux trouvé
        return 0
    return 1 if pos > neg else 0

def llm_score(review):
    response = client.messages.create(
        model="claude-haiku-4-5", max_tokens=120,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": cot_prompt(review)}])
    text = next((block.text for block in response.content if block.type == "text"), "")
    return parse_label(text)

### Lancer le scorer LLM (réservé à l'instructeur)

Pour que la comparaison soit **juste**, le LLM doit voir **tout le jeu de test**, comme les autres modèles. C'est donc à l'**instructeur** de lancer ce scorer **avec sa propre clé**, sur l'ensemble du test (aucune limite sur le nombre d'appels).

**Ordre de grandeur du coût** (Claude Haiku, ~1 $/million de tokens en entrée, ~5 $/million en sortie). Avec un prompt court et une sortie limitée à 120 tokens, les ~800 critiques du test reviennent à **environ 0,3 à 0,7 $ US**. Négligeable pour une démo, mais c'est exactement pourquoi on ne laisse pas une clé partagée tourner en boucle.

Les appels sont lancés **en parallèle** (pool de threads) pour aller bien plus vite ; `pool.map` garde l'ordre des résultats. La cellule ci-dessous est **entièrement commentée** : ne la décommente que si tu es l'instructeur, avec ta propre clé.

In [ ]:
# ======================================================================
# !!!  APPEL AU LLM : NE PAS EXÉCUTER AVEC UNE CLÉ PARTAGÉE / DU COURS  !!!
# !!!  Réservé à l'INSTRUCTEUR, avec SA PROPRE clé.                      !!!
# !!!  ~800 appels API facturés (compter ~0,3 à 0,7 $ US).             !!!
# ======================================================================

# Évaluation JUSTE sur TOUT le test (comme les autres modèles, sans limite).
# Appels EN PARALLÈLE : ~800 requêtes lancées par lots de 8 threads -> bien
# plus rapide qu'une boucle en série. pool.map conserve l'ordre des résultats.

# from concurrent.futures import ThreadPoolExecutor
#
# with ThreadPoolExecutor(max_workers=8) as pool:
#     y_pred_llm = list(pool.map(llm_score, test["text"].tolist()))
#
# evaluate("7. LLM chain-of-thought (Claude Haiku 4.5)",
#          y_test, np.array(y_pred_llm), n=len(test))

## Le verdict : tableau comparatif

On rassemble accuracy et F1 de tous les modèles exécutés. (Le modèle 7 n'apparaît que si l'instructeur a lancé la cellule précédente.)

In [ ]:
comparison = pd.DataFrame(results).T.sort_values("f1", ascending=False)
comparison

In [ ]:
ax = comparison[["accuracy", "f1"]].plot(kind="barh", figsize=(9, 5))
ax.set_xlabel("score"); ax.set_xlim(0, 1)
ax.invert_yaxis(); ax.set_title("Comparaison des approches (test Allociné)")
plt.tight_layout(); plt.show()

## Ce qu'on observe (en général)

- Les **baselines classiques** (1-3) sont étonnamment solides sur une tâche de sentiment binaire : sac de mots et TF-IDF capturent beaucoup de signal, pour un coût dérisoire.
- L'**encodeur gelé** (4) n'est pas toujours meilleur que TF-IDF : un transformer non adapté ne bat pas forcément un bon vieux comptage de mots.
- Le **fine-tuning** (5-6) est là que les transformers prennent l'avantage : plus on adapte l'encodeur, meilleurs sont les embeddings pour *notre* tâche.
- Le **LLM** (7) atteint souvent un excellent score **sans aucun entraînement**, mais il est **lent, facturé**, et c'est une grosse boîte noire pour une tâche qu'un modèle minuscule règle très bien.

**La leçon** : le modèle le plus sophistiqué n'est pas toujours le bon choix. On arbitre entre **performance**, **coût**, **rapidité** et **interprétabilité** selon le problème. Pour classer du sentiment binaire, un TF-IDF suffit souvent ; on réserve le fine-tuning et les LLM aux tâches où le contexte fin est vraiment décisif.

## Que choisir selon la quantité de données étiquetées ?

Un petit aide-mémoire. La question clé n'est pas « quel est le modèle le plus puissant ? » mais « combien d'exemples **étiquetés** ai-je, et combien de textes devrai-je classer ? ».

| Données étiquetées | Approche conseillée | Pourquoi |
|---|---|---|
| **Aucune** | **LLM** (zero-shot) ou modèle pré-entraîné clé en main | Rien à entraîner : le LLM comprend la tâche depuis la consigne |
| **Quelques dizaines** | **LLM** avec quelques exemples dans le prompt (*few-shot*) | Trop peu pour entraîner sans surapprendre |
| **Quelques centaines** | **TF-IDF + régression / ridge** (modèles 2-3) | Solide, quasi gratuit, interprétable ; le fine-tuning n'apporte pas encore grand-chose |
| **Quelques milliers et +** | **Fine-tuning de CamemBERT** (modèles 5-6) | L'encodeur adapté prend l'avantage sur le contexte et les nuances |
| **Beaucoup + gros volume à classer en continu** | **Fine-tuning de CamemBERT** (petit modèle) | Entraîné une fois, puis inférence **rapide et peu chère** à grande échelle |

**Et le LLM dans tout ça ?** Imbattable quand on a **peu ou pas** de données, ou pour prototyper vite. Mais il est **lent et facturé à chaque appel** : pour classer des millions de textes en routine, un petit modèle fine-tuné coûte bien moins cher. Règle de poche : **peu de données → LLM ; beaucoup de données + gros volume → fine-tuning ; entre les deux → TF-IDF d'abord, fine-tuning si besoin.**